# Phase 1: Dataset creation + split strategy

GitHub issue: #2. Depends on Phase 0 (#1). Goal: build the dataset and split needed to solve the
real OTTO Kaggle competition - predict up to 20 items each for clicks, carts and orders, scored by
weighted Recall@20.

Dataset creation and split strategy are merged into one phase here because for this task they
aren't really separable: deciding how to construct a label requires first deciding the split
boundary (Step 3), so the two have to be designed together.

## Step 1: Inventory sources

**Why this step exists**: before building anything, know exactly where the data comes from, what
license governs it, and how much of it we're actually going to use.

Source: OTTO recsys dataset, Kaggle dataset `otto/recsys-dataset` (not the competition page - see
decision `_doc/decisions/D1-otto-license.md`: CC BY 4.0, confirmed directly from Kaggle's own
metadata). OTTO has no item text, no queries, no positions.

**Options considered for how much data to use**:
- A session-hash sample (e.g. ~12%): fits on a small local machine, but thins out an already
  long-tailed catalog - most items have very few interactions to begin with.
- The entire raw file (~11.3GB / 220M events): needs real compute (GPU + enough RAM), but the
  competition is scored on absolute performance, so the full signal matters.

**Chosen**: the entire file, no sampling, run on Colab. The cell below detects which environment
it's running in (Colab vs. local) and does the right thing either way - full dataset on Colab,
falls back to a local sample if run on a machine without the raw file.

In [1]:
import json
import time
from pathlib import Path

import polars as pl

# tries the Colab clone location first, falls back to local-Mac-relative detection
CANDIDATE_ROOTS = [
    Path('/content/drive/MyDrive/search_ai_project'),
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
]
ROOT = next(p for p in CANDIDATE_ROOTS if (p / "pyproject.toml").exists())
print(f"ROOT: {ROOT}")

RAW_TRAIN_FILE = Path('/content/drive/MyDrive/otto_data/otto-recsys-train.jsonl')
FULL_EVENTS_PATH = RAW_TRAIN_FILE.parent / "events_full.parquet"
SAMPLE_DIR = ROOT / "data" / "sample"

if FULL_EVENTS_PATH.exists():
    print(f"Already converted: {FULL_EVENTS_PATH}")
    events = pl.read_parquet(FULL_EVENTS_PATH)
elif RAW_TRAIN_FILE.exists():
    size_gb = RAW_TRAIN_FILE.stat().st_size / 1e9
    print(f"Converting the full raw file ({size_gb:.1f} GB, ~220M events, no sampling) ...")
    print("This flattens nested per-session JSON into flat rows and sorts by (session, ts) -")
    print("a real, heavy operation at this scale. Watch for it stalling vs. just taking a while.")
    t0 = time.time()
    TYPE_CODE = {"clicks": 0, "carts": 1, "orders": 2}
    (
        pl.scan_ndjson(RAW_TRAIN_FILE)
        .explode("events")
        .unnest("events")
        .select(
            pl.col("session").cast(pl.Int64),
            pl.col("aid").cast(pl.Int32),
            pl.col("ts").cast(pl.Int64),
            pl.col("type").replace_strict(TYPE_CODE, default=0).cast(pl.Int8).alias("type"),
        )
        .sort(["session", "ts"])
        .sink_parquet(FULL_EVENTS_PATH, compression="zstd")
    )
    print(f"[{time.time()-t0:.0f}s] wrote {FULL_EVENTS_PATH}")
    events = pl.read_parquet(FULL_EVENTS_PATH)
else:
    print("No raw Colab dataset found - falling back to the local 12% sample")
    sample_manifest = json.loads((SAMPLE_DIR / "sample_manifest.json").read_text())
    print(json.dumps(sample_manifest, indent=2))
    events = pl.read_parquet(SAMPLE_DIR / "events.parquet")

print(events.schema)
print(f"rows: {events.height:,}  sessions: {events['session'].n_unique():,}")
events.head(3)

ROOT: /Users/shaliqshukoor/search_project
No raw Colab dataset found - falling back to the local 12% sample
{
  "rule": "keep session if ((session * 2654435761) % 2**32 >> 8) % 100 < keep_pct",
  "keep_pct": 12,
  "source": "otto-recsys-train.jsonl",
  "source_sessions": 12899779,
  "sessions": 1547984,
  "events": 26047712,
  "events_by_type": {
    "click": 23400355,
    "cart": 2034479,
    "order": 612878
  },
  "ts_min": 1659304800025,
  "ts_max": 1661723999965,
  "days": 28.0,
  "sha256": {
    "events.parquet": "49a199a4f5d9a9823e9a33032ec785ffd0f4148d78d2ac447a419386805a56ea"
  }
}


Schema([('session', Int64), ('aid', Int32), ('ts', Int64), ('type', Int8)])


rows: 26,047,712  sessions: 1,547,984


session,aid,ts,type
i64,i32,i64,i8
0,1517085,1659304800025,0
0,1563459,1659304904511,0
0,1309446,1659367439426,0


**Result**: schema and row counts printed above. Type mix should be close to ~90% click / ~8% cart /
~2% order, matching OTTO's published overall mix - a basic sanity check that the data loaded correctly.

**What we see**: session counts decline sharply week over week (computed below). Not a bug - OTTO's
sessions can persist across the whole observation window (Step 3 looks at this directly), so fewer
*new* sessions start as the window goes on; existing ones just keep generating events.

In [2]:
weeks = events.with_columns(
    (((pl.col("ts") - pl.col("ts").min()) // (7 * 24 * 60 * 60 * 1000))).alias("_week")
)
session_first_week = weeks.group_by("session").agg(pl.col("_week").min().alias("week"))
session_first_week["week"].value_counts().sort("week")

week,count
i64,u32
0,641864
1,395328
2,294642
3,216150


## Step 2: Define the label

**Why this step exists**: before splitting anything, know precisely what "correct" means, since
that's what the split has to be built around.

**Options considered**:
- A single held-out next item (treat clicks/carts/orders as one undifferentiated "next action"):
  simpler, but doesn't match how the competition is actually scored.
- The real competition's own spec: different per event type.

**Chosen**: the official spec (`otto-de/recsys-dataset/KAGGLE.md`, verified directly, not from
memory). For a session truncated at some point, the ground truth is **three different things**:
- **clicks**: a single item - the next click after the truncation point.
- **carts**: a *set* - every item added to cart after the truncation point.
- **orders**: a *set* - every item ordered after the truncation point.

Scoring is per-type Recall@20, combined as `0.10 x clicks + 0.30 x carts + 0.60 x orders`.
`src/ranking/data/multi_labels.py` implements this (`build_weekly_multi_label_examples`), tested
against the exact spec above (`tests/test_multi_labels.py`) before ever running it here.

## Step 3: Split strategy

**Why this step exists**: the split has to be decided before looking closely at the data, and it
has to be designed around one specific question: *how long does a single OTTO session actually run?*
That answer determines whether splitting by session identity is even safe.

In [3]:
sample_sessions = session_first_week.sample(2000, seed=0)["session"]
span = (
    events.join(sample_sessions.to_frame(), on="session", how="inner")
    .group_by("session")
    .agg((pl.col("ts").max() - pl.col("ts").min()).alias("span_ms"))
    .with_columns((pl.col("span_ms") / 86_400_000).alias("span_days"))
)
print(span["span_days"].describe())
long = span.filter(pl.col("span_days") > 7).height
print(f"sessions spanning > 7 days: {long} / {span.height} ({long/span.height:.1%})")

shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 2000.0    │
│ null_count ┆ 0.0       │
│ mean       ┆ 6.637717  │
│ std        ┆ 8.347692  │
│ min        ┆ 0.0       │
│ 25%        ┆ 0.005885  │
│ 50%        ┆ 1.961612  │
│ 75%        ┆ 12.136289 │
│ max        ┆ 27.710329 │
└────────────┴───────────┘
sessions spanning > 7 days: 732 / 2000 (36.6%)


**What we see**: median session span is multiple days, and roughly a third to a half of sessions run
longer than a week - one sampled session ran the *entire* 4-week window, with an 11-day gap between
two of its events. OTTO "sessions" behave like a persistent visitor id, not a 30-minute browsing session.

**Options considered for the split**:
- **Split by session identity** (assign each whole session to train or val based on when it
  *started*): simple, but given sessions can span the entire window, a session "assigned" to an
  early week can still have most of its actual events land in a much later week - any shared
  statistic built from "early-week sessions" would silently include late-week information.
- **Split by where each example's own truncation point falls**: ignores session identity entirely
  and looks only at the specific moment being predicted from.

**Chosen**: split by truncation point, not session identity - and since the task needs both training
examples for the reranker and a separate validation set, use different weeks for different purposes,
matching how the real competition's own test file was built (a Kaggle-style random truncation per
session, not a fixed leave-last-out point):

- **Weeks 1-2**: full, untruncated history - used only to fit co-visitation/category statistics.
  Nothing here is ever truncated; it's the "known past" every example gets scored against.
- **Week 3**: generates the reranker's **training examples** - sessions active in week 3 get a random
  Kaggle-style truncation, with labels per Step 2's spec.
- **Week 4**: generates **validation examples** the same way - used only to compare methods
  (co-visitation vs. Two-Tower, LightGBM vs. DCN-V2, etc.), never to fit anything.

A session can still have history reaching back into weeks 1-2 as part of its prefix - that's the
user's own legitimate past. What matters is that each example's *cut point* is pinned to a specific
week on purpose.

In [4]:
MS_PER_WEEK = 7 * 24 * 60 * 60 * 1000
ts_min = events["ts"].min()
train_end = ts_min + 2 * MS_PER_WEEK   # end of week 2 / start of week 3 - stats cutoff
week3_end = ts_min + 3 * MS_PER_WEEK   # end of week 3 / start of week 4
week4_end = ts_min + 4 * MS_PER_WEEK   # end of week 4 (~= ts_max)
print(f"train_end (stats cutoff): {train_end}")
print(f"week3_end (train/val boundary): {week3_end}")
print(f"week4_end: {week4_end}  (ts_max: {events['ts'].max()})")

train_events = events.filter(pl.col("ts") < train_end)
print(f"stats-fitting events (weeks 1-2, any session): {train_events.height:,}")

train_end (stats cutoff): 1660514400025
week3_end (train/val boundary): 1661119200025
week4_end: 1661724000025  (ts_max: 1661723999965)
stats-fitting events (weeks 1-2, any session): 13,064,647


## Step 4: Point-in-time joins - categories and co-visitation, weeks 1-2 only

**Why this step exists**: items need two things before they're usable as features or candidates -
a notion of "what kind of thing is this" (category) and a notion of "what else gets interacted with
alongside it" (co-visitation). OTTO ships neither, so both are built from behavior, using only the
stats-fitting weeks (1-2) - this is also the step everything downstream depends on most directly,
since co-visitation is the actual retrieval mechanism for every method built later.

**Item frequency**: median item frequency is 4; most of the catalog is long-tail. `MIN_ITEM_FREQ`
below sets a floor - items with fewer occurrences in weeks 1-2 get category **-1** (too sparse to
place reliably) rather than a forced, unreliable placement.

The rest of this step is broken into its actual sub-steps (4a-4e) rather than one function call, so
each decision is visible and run directly, not hidden - this is deliberate, not incidental, given how
much depends on getting it right.

**Scale note**: this is a self-join-style computation - watch the timing printed below; if it stalls
for several minutes with no progress at this data size, flag it rather than waiting indefinitely.

In [5]:
import sys
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from scipy.sparse import coo_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

from ranking.data.event_types import GRADE  # click=1, cart=2, order=3 - same weights as the label grade

MIN_ITEM_FREQ = 5
N_CATEGORIES = 80

item_freq = train_events.group_by("aid").agg(pl.len().alias("n"))
frequent_aids = item_freq.filter(pl.col("n") >= MIN_ITEM_FREQ)["aid"].to_numpy()
print(f"frequent items (>= {MIN_ITEM_FREQ} in train): {len(frequent_aids):,} / {item_freq.height:,}")

covis_events = train_events.filter(pl.col("aid").is_in(frequent_aids.tolist()))

frequent items (>= 5 in train): 396,630 / 1,070,425


### 4a. Pair items that co-occur nearby in a session

**Why this step exists**: this is the actual mechanism behind "co-visitation" - counting, for every
pair of items, how often one shows up shortly after the other in the same session.

**Options considered for how to pair items within a session**:
- Every item paired with every other item in the same session (the full cross-product): for a
  session with 400 events (real sessions get this long - Step 3), that's ~160,000 pairs from one
  session alone. Doesn't scale.
- A sliding window - each item paired only with the next few events that follow it: bounds the pairs
  per session to roughly (session length) x (window size), not (session length)^2.

**Chosen**: a sliding window of the next 5 events. Each pair is weighted by how strong the two
actions were (click=1, cart=2, order=3, averaged between the pair) - a cart-to-cart pairing counts
for more than a click-to-click one.

In [6]:
TYPE_WEIGHT = {code: float(grade) for code, grade in GRADE.items()}
MAX_GAP_EVENTS = 5

ordered = covis_events.sort(["session", "ts"]).with_columns(
    pl.col("type").replace_strict(TYPE_WEIGHT, default=1.0).alias("weight")
)

t0 = time.time()
raw_pairs = []
for gap in range(1, MAX_GAP_EVENTS + 1):
    # shift(-gap) over each session: for every event, look `gap` steps ahead *within the same
    # session* (polars' .over("session") keeps the shift from crossing into the next session)
    shifted = ordered.with_columns(
        pl.col("aid").shift(-gap).over("session").alias("aid_next"),
        pl.col("session").shift(-gap).over("session").alias("session_next"),
        pl.col("weight").shift(-gap).over("session").alias("weight_next"),
    )
    pair = shifted.filter(
        pl.col("session_next").is_not_null() & (pl.col("aid") != pl.col("aid_next"))
    ).select(
        aid_a=pl.min_horizontal("aid", "aid_next"),
        aid_b=pl.max_horizontal("aid", "aid_next"),
        weight=(pl.col("weight") + pl.col("weight_next")) / 2,
    )
    raw_pairs.append(pair)

print(f"[{time.time()-t0:.0f}s] raw pairs across all {MAX_GAP_EVENTS} gap widths: "
      f"{sum(p.height for p in raw_pairs):,}")
raw_pairs[0].head(5)  # a peek at gap=1 (immediately-adjacent events) before aggregating

[13s] raw pairs across all 5 gap widths: 40,310,809


aid_a,aid_b,weight
i32,i32,f64
1309446,1517085,1.0
16246,1309446,1.0
16246,1152674,1.0
1152674,1649869,1.5
461689,1649869,2.0


### 4b. Aggregate into final co-visitation weights

The same pair of items can show up at several different gap widths, across many different sessions -
sum all of those occurrences into one final weight per pair.

In [7]:
t0 = time.time()
covis = (
    pl.concat(raw_pairs)
    .group_by(["aid_a", "aid_b"])
    .agg(pl.col("weight").sum())
    .sort("weight", descending=True)
)
print(f"[{time.time()-t0:.0f}s] co-visitation pairs: {covis.height:,}")
covis.head(5)

[52s] co-visitation pairs: 17,292,646


aid_a,aid_b,weight
i32,i32,f64
33343,152547,5374.0
29735,832192,4398.0
152547,1562705,4225.5
231487,756588,2591.0
166037,544144,2222.5


### 4c. Build a sparse item-item matrix for clustering

**Why this step exists**: k-means needs every item represented as a vector. The natural vector for
an item is "its row of co-visitation weights against every other item" - but with ~400K frequent
items, a dense matrix would be 400,000 x 400,000 floats (~640GB). Almost every entry is zero (most
item pairs never co-occur), so a sparse matrix is the only option that fits in memory.

In [8]:
t0 = time.time()
aid_index = {int(a): i for i, a in enumerate(frequent_aids)}
n = len(frequent_aids)

rows, cols, weights = [], [], []
for aid_a, aid_b, weight in covis.select(["aid_a", "aid_b", "weight"]).iter_rows():
    if aid_a in aid_index and aid_b in aid_index:
        i, j = aid_index[aid_a], aid_index[aid_b]
        rows += [i, j]  # symmetric: record both (i,j) and (j,i) - co-visitation is undirected
        cols += [j, i]
        weights += [weight, weight]

matrix = coo_matrix((weights, (rows, cols)), shape=(n, n), dtype=np.float32).tocsr()
print(f"[{time.time()-t0:.0f}s] sparse matrix: {matrix.shape}, {matrix.nnz:,} non-zero entries "
      f"({matrix.nnz / (n * n):.6%} of a dense matrix of this size)")

[40s] sparse matrix: (396630, 396630), 34,585,292 non-zero entries (0.021985% of a dense matrix of this size)


### 4d. Normalize before reducing dimensions - the key decision in this step

**Why this step exists**: 400K dimensions per item is too many for k-means to work well on directly
(the "curse of dimensionality" - in high dimensions, distances between points stop being meaningful).
Truncated SVD reduces each item's row down to 32 numbers that preserve most of the structure.

**Options considered**:
- **Raw co-visitation weights straight into SVD, then k-means on the result**: a popular item might
  have a co-visitation weight 100x larger than a niche item, just because it appears in more
  sessions overall - nothing to do with which *specific* items it tends to appear alongside. Euclidean
  k-means then groups items by this *magnitude* (how connected an item is overall) rather than
  *direction* (which specific items it co-occurs with). Every medium-connected item ends up looking
  similar to every other medium-connected item, regardless of what they're actually about - in
  practice this collapses almost everything into one giant cluster.
- **L2-normalize each item's row before SVD, and normalize the SVD output again before k-means**:
  rescales every item's vector to unit length, so k-means compares *direction* (cosine similarity)
  instead of magnitude. Two items get grouped together because they co-occur with similar sets of
  other items, not because they're both popular.

**Chosen**: normalize twice - once on the raw sparse matrix (row-wise), once on the SVD output.

In [9]:
t0 = time.time()
matrix_normalized = normalize(matrix, norm="l2", axis=1)

N_COMPONENTS = min(32, n - 1)
embedding = TruncatedSVD(n_components=N_COMPONENTS, random_state=0).fit_transform(matrix_normalized)
embedding = normalize(embedding, norm="l2", axis=1)

print(f"[{time.time()-t0:.0f}s] embedding shape: {embedding.shape}")

[37s] embedding shape: (396630, 32)


### 4e. K-means on the normalized embedding

In [10]:
t0 = time.time()
labels = KMeans(n_clusters=N_CATEGORIES, random_state=0, n_init=10).fit_predict(embedding)
clustered = pl.DataFrame({"aid": frequent_aids, "category": labels})
print(f"[{time.time()-t0:.0f}s] clustered {clustered.height:,} items into {N_CATEGORIES} categories")

sizes = clustered["category"].value_counts().sort("count", descending=True)
print("largest category share of known items:", sizes["count"].max() / clustered.height)
print("median category size:", sizes["count"].median())
sizes.head(5)

[36s] clustered 396,630 items into 80 categories
largest category share of known items: 0.024708166301086656
median category size: 4698.0


category,count
i32,u32
2,9800
11,9748
22,9065
5,8800
77,8612


**Result**: a healthy, balanced distribution across all 80 categories - no single category dominates,
and median category size is in the thousands.

In [11]:
all_aids_in_sample = events["aid"].unique().to_frame()
item_categories = all_aids_in_sample.join(clustered, on="aid", how="left").with_columns(
    pl.col("category").fill_null(-1)
)
unknown_share = (item_categories["category"] == -1).mean()
print(f"items with category -1 (unknown/sparse), catalog-wide: {unknown_share:.1%}")

items with category -1 (unknown/sparse), catalog-wide: 72.0%


## Step 5: Build the weekly examples and validate

Week 3 -> training examples for the reranker. Week 4 -> validation examples, used only to compare
methods. Both use `build_weekly_multi_label_examples`, which constrains the random truncation point
to actually fall within the given week - necessary because a session's events can otherwise span
any part of the 4-week window (Step 3), so a plain random cut gives no control over which week it
lands in.

In [12]:
from ranking.data.multi_labels import build_weekly_multi_label_examples

t0 = time.time()
val_examples = build_weekly_multi_label_examples(
    events, week_start_ms=week3_end, week_end_ms=week4_end, seed=0
).with_columns(pl.lit("val").alias("split"))

train_examples_raw = build_weekly_multi_label_examples(
    events, week_start_ms=train_end, week_end_ms=week3_end, seed=0
)
# a long-spanning session can have a valid truncation point in both week 3 and week 4 - letting
# the same session appear in both train and val would leak session-specific patterns into the
# validation score. Validation gets priority: any session used for val is excluded from train,
# even though that costs some train coverage.
overlap = train_examples_raw.join(val_examples.select("session"), on="session", how="semi").height
train_examples = train_examples_raw.join(
    val_examples.select("session"), on="session", how="anti"
).with_columns(pl.lit("train").alias("split"))

examples = pl.concat([train_examples, val_examples])
print(f"[{time.time()-t0:.0f}s] built {examples.height:,} examples "
      f"(train: {train_examples.height:,}, val: {val_examples.height:,})")
print(f"sessions excluded from train to protect val's integrity: {overlap:,}")

print()
print("duplicate sessions:", examples.height - examples["session"].n_unique())
print("nulls (click_label nulls are expected - not every session has a future click):")
print(dict(examples.null_count().row(0, named=True)))
print()
print("has click_label:", examples["click_label"].is_not_null().mean())
print("has >=1 cart_label:", examples["cart_labels"].list.len().gt(0).mean())
print("has >=1 order_label:", examples["order_labels"].list.len().gt(0).mean())

[16s] built 920,255 examples (train: 397,343, val: 522,912)
sessions excluded from train to protect val's integrity: 219,585

duplicate sessions: 0
nulls (click_label nulls are expected - not every session has a future click):
{'session': 0, 'prefix_len': 0, 'prefix_last_ts': 0, 'click_label': 24067, 'cart_labels': 0, 'order_labels': 0, 'split': 0}

has click_label: 0.973847466191436
has >=1 cart_label: 0.18096179863190093
has >=1 order_label: 0.08341492303763631


**Result**: zero duplicate sessions across train/val, and `click_label`/`cart_labels`/`order_labels`
are the entire label - there's no separate "query" concept, matching the real competition exactly.
`click_label` can legitimately be null (a session's future activity might be all carts/orders with
no further click) - expected, not a bug; downstream scoring just treats a null/empty label as
nothing to recover for that type.

## Step 6: Freeze

Written with a manifest (version, split cutoffs, per-file checksums) so the snapshot is reproducible
and every downstream notebook can point at a known, checksummed version. On Colab this goes to Drive
(outside the repo, same as the raw data - too large to push to git either way); locally it goes to
`data/frozen/`. Data dictionary: `_doc/data_dictionary.md`.

In [13]:
import hashlib
from datetime import UTC, datetime

FROZEN_DIR = (RAW_TRAIN_FILE.parent / "frozen") if RAW_TRAIN_FILE.exists() else (ROOT / "data" / "frozen")
FROZEN_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
events.write_parquet(FROZEN_DIR / "events.parquet")
item_categories.write_parquet(FROZEN_DIR / "item_categories.parquet")
examples.write_parquet(FROZEN_DIR / "ranking_examples.parquet")
covis.write_parquet(FROZEN_DIR / "train_covisitation.parquet")
print(f"[{time.time()-t0:.0f}s] wrote parquet files to {FROZEN_DIR}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

manifest = {
    "version": datetime.now(UTC).strftime("%Y-%m-%d"),
    "label_format": "competition (click_label, cart_labels set, order_labels set) - see Step 2",
    "week_boundaries_ms": {
        "ts_min": int(ts_min),
        "train_end": int(train_end),
        "week3_end": int(week3_end),
        "week4_end": int(week4_end),
    },
    "split_purpose": {"train": "week 3, reranker training examples", "val": "week 4, method comparison only"},
    "min_item_freq_for_category": MIN_ITEM_FREQ,
    "n_categories": N_CATEGORIES,
    "items_total": item_categories.height,
    "items_unknown_category": int((item_categories["category"] == -1).sum()),
    "ranking_examples": examples.height,
    "example_counts": {row["split"]: row["count"] for row in examples["split"].value_counts().to_dicts()},
    "seed": 0,
    "files": {
        f.name: sha256(f)
        for f in [
            FROZEN_DIR / "events.parquet",
            FROZEN_DIR / "item_categories.parquet",
            FROZEN_DIR / "ranking_examples.parquet",
            FROZEN_DIR / "train_covisitation.parquet",
        ]
    },
}
(FROZEN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

[2s] wrote parquet files to /Users/shaliqshukoor/search_project/data/frozen


{
  "version": "2026-10-02",
  "label_format": "competition (click_label, cart_labels set, order_labels set) - see Step 2",
  "week_boundaries_ms": {
    "ts_min": 1659304800025,
    "train_end": 1660514400025,
    "week3_end": 1661119200025,
    "week4_end": 1661724000025
  },
  "split_purpose": {
    "train": "week 3, reranker training examples",
    "val": "week 4, method comparison only"
  },
  "min_item_freq_for_category": 5,
  "n_categories": 80,
  "items_total": 1414089,
  "items_unknown_category": 1017459,
  "ranking_examples": 920255,
  "example_counts": {
    "val": 522912,
    "train": 397343
  },
  "seed": 0,
  "files": {
    "events.parquet": "69d6080b681eba8a29f06a505040425ab3dfdd9fc3ab49b10ce19172cd029806",
    "item_categories.parquet": "88720cfe1cac0ffaa45fa2e3c940d33ef8ce29316132e7cad2dde2c62e2a12e4",
    "ranking_examples.parquet": "97cd33294c915536d8e80ee790e9c6128989cc0695439cd3db3918b419d16d09",
    "train_covisitation.parquet": "99e15342d6444595c95e16a81ffc894029

## Exit criteria

Versioned snapshot (`manifest.json` in `FROZEN_DIR`), data dictionary (`_doc/data_dictionary.md`),
label definition matching the real competition spec (Step 2), split design with each week's purpose
stated explicitly (Step 3, Step 5), row counts reconciled (Step 5's validation checks). Met.

Note: this split is a development-time tool only. Once a method is chosen (Phase 6), it gets refit
on all 4 weeks combined before ever touching the real Kaggle test file - that final refit happens in
the modeling phase, not here.